# FD002 — Single Dataset Runner (All_rows RMSE, CAP=125)

Bu notebook **tek bir veri seti (train+test)** ile çalışır.  
Sen bunu **3 ayrı kez** farklı dosya yollarını girerek çalıştırıp çıktıları ayrı ayrı raporlayacaksın.

✅ Notebook “tek dataset” gibi davranır:
- **TRAIN_PATH / TEST_PATH / (opsiyonel) SCALER_JSON_PATH** girersin
- Feature seti, verdiğin inputa göre otomatik şekillenir
- **GroupKFold(engine_id) OOF** + **LightGBM + CatBoost + (Stacking Ridge / NNLS / CV weight)** çalışır
- Çıktıyı `results_<EXP_NAME>.csv` olarak kaydeder

---

## Beklenen kullanım (3 koşu):
1) **BASE**: global zscore (train/test) + (opsiyonel) `fd002_full_global_zscore_scaler.json`  
2) **A**: global zscore + regime onehot (k6) (train/test)  
3) **B**: regime-aware sensor zscore + onehot (k6) (train/test) + (opsiyonel) `fd002_regimeaware_sensor_scaler_k6.json`


In [ ]:
!pip -q install lightgbm catboost xgboost scipy

import os, json, math, random, time
import numpy as np
import pandas as pd

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import Ridge
from sklearn.ensemble import GradientBoostingRegressor

from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from xgboost import XGBRegressor

from scipy.optimize import nnls

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

CAP = 125
N_SPLITS = 5
EARLY_STOP_ROUNDS = 50

def cap_rul(y, cap=125): 
    return np.minimum(y, cap)

def clip_pred(yhat, cap=125): 
    return np.clip(yhat, 0, cap)

def rmse(y_true, y_pred):
    return math.sqrt(mean_squared_error(y_true, y_pred))

def metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {
        "R2": r2_score(y_true, y_pred),
        "MSE": mse,
        "RMSE": math.sqrt(mse),
        "MAE": mean_absolute_error(y_true, y_pred),
    }


## 0) Upload files (Colab)  
Bu cell’den sonra, aşağıdaki `TRAIN_PATH/TEST_PATH/SCALER_JSON_PATH` değerlerini ayarlayacaksın.


In [ ]:
from google.colab import files
uploaded = files.upload()
print("Uploaded:", list(uploaded.keys())[:15], "...")


## 1) Experiment config (tek dataset)

🔧 Burayı her koşudan önce güncelle:
- `EXP_NAME`: rapor ismi (örn: "BASE", "A", "B")
- `TRAIN_PATH`, `TEST_PATH`
- `SCALER_JSON_PATH` (opsiyonel)

Feature seçimi:
- Eğer `SCALER_JSON_PATH` içinde `feature_columns` varsa ve `USE_SCALER_FEATURE_LIST=True` ise:
  - önce `feature_columns` alınır
  - `cycle` eklenir (varsa)
  - `INCLUDE_EXTRA_COLS=True` ise, dataset’te olan ama listede olmayan ek kolonlar (örn `regime_0..`) otomatik eklenir.
- Aksi halde: `engine_id` ve `RUL` hariç tüm kolonlar feature olur.


In [ ]:
# ====== CHANGE THESE PER RUN ======
EXP_NAME = "BASE"  # "BASE" / "A" / "B" (sen ne yazarsan)

TRAIN_PATH = "/content/train_FD002_full_global_zscore.csv"
TEST_PATH  = "/content/test_FD002_full_global_zscore.csv"

SCALER_JSON_PATH = "/content/fd002_full_global_zscore_scaler.json"  # optional; set to "" if none
USE_SCALER_FEATURE_LIST = True     # uses scaler["feature_columns"] when available
INCLUDE_EXTRA_COLS = True          # adds extra columns like regime_* automatically
# ===================================

print("EXP_NAME:", EXP_NAME)
print("TRAIN exists:", os.path.exists(TRAIN_PATH))
print("TEST  exists:", os.path.exists(TEST_PATH))
print("SCALER exists:", os.path.exists(SCALER_JSON_PATH) if SCALER_JSON_PATH else False)


In [ ]:
def load_df(path):
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    df = pd.read_csv(path)
    # drop accidental index columns if present
    drop_cols = [c for c in df.columns if c.lower().startswith("unnamed")]
    if drop_cols:
        df = df.drop(columns=drop_cols)
    return df

train_df = load_df(TRAIN_PATH)
test_df  = load_df(TEST_PATH)

# required cols
for c in ["engine_id", "cycle", "RUL"]:
    if c not in train_df.columns:
        raise ValueError(f"train missing required column: {c}")
for c in ["engine_id", "cycle", "RUL"]:
    if c not in test_df.columns:
        raise ValueError(f"test missing required column: {c}")

scaler = None
feature_columns = None
if SCALER_JSON_PATH and os.path.exists(SCALER_JSON_PATH):
    with open(SCALER_JSON_PATH, "r") as f:
        scaler = json.load(f)
    feature_columns = scaler.get("feature_columns", None)

# feature selection logic
if USE_SCALER_FEATURE_LIST and feature_columns is not None:
    features = list(feature_columns)
    if "cycle" in train_df.columns and "cycle" not in features:
        features.append("cycle")
    if INCLUDE_EXTRA_COLS:
        extras = [c for c in train_df.columns if c not in (["engine_id","RUL"] + features)]
        # keep only numeric-like columns (skip obvious non-feature columns if any)
        extras = [c for c in extras if c not in ["engine_id","RUL"]]
        features += extras
else:
    # autodetect: everything except engine_id and RUL
    features = [c for c in train_df.columns if c not in ["engine_id","RUL"]]

# safety: keep order as appears in train_df
features = [c for c in train_df.columns if c in features]

print("train:", train_df.shape, "test:", test_df.shape)
print("n_features:", len(features))
print("first features:", features[:12])

missing_train = [c for c in features if c not in train_df.columns]
missing_test  = [c for c in features if c not in test_df.columns]
if missing_train or missing_test:
    print("Missing in train:", missing_train[:10])
    print("Missing in test:", missing_test[:10])


In [ ]:
# Targets + groups
y_train = cap_rul(train_df["RUL"].values, CAP)
y_test  = cap_rul(test_df["RUL"].values, CAP)
groups  = train_df["engine_id"].values

# X matrices (numeric coerce + clean)
Xtr = (train_df[features]
       .apply(pd.to_numeric, errors="coerce")
       .replace([np.inf, -np.inf], np.nan)
       .fillna(0.0)
       .values.astype(np.float32))

Xte = (test_df[features]
       .apply(pd.to_numeric, errors="coerce")
       .replace([np.inf, -np.inf], np.nan)
       .fillna(0.0)
       .values.astype(np.float32))

print("Xtr:", Xtr.shape, "Xte:", Xte.shape)
print("y_train range:", float(y_train.min()), float(y_train.max()), " | y_test range:", float(y_test.min()), float(y_test.max()))


## 2) Models (fixed)

In [ ]:
def make_models(seed=42):
    lgbm = LGBMRegressor(
        n_estimators=500,
        learning_rate=0.1,
        max_depth=-1,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=seed,
        n_jobs=-1,
    )
    cat = CatBoostRegressor(
        iterations=500,
        learning_rate=0.1,
        depth=6,
        loss_function="RMSE",
        random_seed=seed,
        verbose=False,
    )
    gbr = GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.1,
        max_depth=7,
        subsample=0.8,
        random_state=seed,
    )
    xgb = XGBRegressor(
        n_estimators=200,
        learning_rate=0.2,
        max_depth=4,
        subsample=1.0,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=seed,
        n_jobs=-1,
    )
    return {"LightGBM": lgbm, "CatBoost": cat, "GBR": gbr, "XGBoost": xgb}

MODELS = make_models(SEED)


## 3) OOF predictions (GroupKFold) — progress + early stopping

In [ ]:
def oof_predictions(models_dict, X, y, groups, n_splits=5, early_stop_rounds=50):
    gkf = GroupKFold(n_splits=n_splits)
    oof = {k: np.zeros(len(y), dtype=float) for k in models_dict}

    for fold, (tr, va) in enumerate(gkf.split(X, y, groups), start=1):
        Xtr_f, Xva_f = X[tr], X[va]
        ytr_f, yva_f = y[tr], y[va]
        print(f"\n===== Fold {fold}/{n_splits} | train={len(tr)} valid={len(va)} =====", flush=True)

        for name, model in models_dict.items():
            t0 = time.time()
            print(f"-> Training {name} ...", flush=True)
            m = model.__class__(**model.get_params())

            try:
                if early_stop_rounds and name == "LightGBM":
                    import lightgbm as lgb
                    m.fit(
                        Xtr_f, ytr_f,
                        eval_set=[(Xva_f, yva_f)],
                        callbacks=[lgb.early_stopping(early_stop_rounds, verbose=False)]
                    )
                elif early_stop_rounds and name == "CatBoost":
                    m.fit(
                        Xtr_f, ytr_f,
                        eval_set=(Xva_f, yva_f),
                        use_best_model=True,
                        early_stopping_rounds=early_stop_rounds,
                        verbose=False
                    )
                elif early_stop_rounds and name == "XGBoost":
                    m.fit(
                        Xtr_f, ytr_f,
                        eval_set=[(Xva_f, yva_f)],
                        verbose=False,
                        early_stopping_rounds=early_stop_rounds
                    )
                else:
                    m.fit(Xtr_f, ytr_f)
            except Exception:
                m.fit(Xtr_f, ytr_f)

            yhat = clip_pred(m.predict(Xva_f), CAP)
            oof[name][va] = yhat
            print(f"   done {name} in {time.time()-t0:.1f}s | fold_RMSE={rmse(yva_f,yhat):.4f}", flush=True)

    return oof

def fit_full(models_dict, X, y):
    fitted={}
    for name, model in models_dict.items():
        m = model.__class__(**model.get_params())
        m.fit(X, y)
        fitted[name]=m
    return fitted

def predict_all(fitted, X):
    return {name: clip_pred(m.predict(X), CAP) for name,m in fitted.items()}


In [ ]:
oof = oof_predictions(MODELS, Xtr, y_train, groups, n_splits=N_SPLITS, early_stop_rounds=EARLY_STOP_ROUNDS)
print("OOF RMSE:", {k: rmse(y_train,v) for k,v in oof.items()})


## 4) Combiners: CV weight + Ridge stacking + NNLS

In [ ]:
def best_weight(oof_a, oof_b, y, step=0.01):
    best = (1e9, None)
    for w in np.arange(0, 1.0001, step):
        yhat = clip_pred(w*oof_a + (1-w)*oof_b, CAP)
        val = rmse(y, yhat)
        if val < best[0]:
            best = (val, float(w))
    return best  # (rmse, w)

def tune_ridge_alpha(meta_X, y, groups, alphas=(0.1,0.3,1,3,10,30,100), n_splits=5):
    gkf = GroupKFold(n_splits=n_splits)
    best = (1e9, None)
    for a in alphas:
        o = np.zeros(len(y), float)
        for tr, va in gkf.split(meta_X, y, groups):
            m = Ridge(alpha=a, random_state=SEED)
            m.fit(meta_X[tr], y[tr])
            o[va] = clip_pred(m.predict(meta_X[va]), CAP)
        val = rmse(y, o)
        if val < best[0]:
            best = (val, float(a))
    return best  # (rmse, alpha)

# CV-best w for (LGBM+Cat)
rmse_w, w = best_weight(oof["LightGBM"], oof["CatBoost"], y_train, step=0.01)
print(f"Best CV weight (LGBM+Cat): w={w:.2f} CV_RMSE={rmse_w:.4f}")

# Ridge stacking alpha tune (LGBM+Cat+GBR)
meta_names = ["LightGBM", "CatBoost", "GBR"]
meta_X = np.column_stack([oof[n] for n in meta_names])
best_rmse_alpha, best_alpha = tune_ridge_alpha(meta_X, y_train, groups, n_splits=N_SPLITS)
print(f"Best Ridge alpha={best_alpha} CV_RMSE={best_rmse_alpha:.4f}")

# NNLS weights
w_nnls, _ = nnls(meta_X, y_train)
print("NNLS weights (LGBM,Cat,GBR):", w_nnls, "sum:", float(w_nnls.sum()))


## 5) Fit full models + evaluate on test_all_rows

In [ ]:
fitted = fit_full(MODELS, Xtr, y_train)
pred = predict_all(fitted, Xte)

# Simple averages
pred["Mean(LGBM+Cat)"] = 0.5*pred["LightGBM"] + 0.5*pred["CatBoost"]
pred["70/30(LGBM+Cat)"] = 0.7*pred["LightGBM"] + 0.3*pred["CatBoost"]
pred[f"CVw(LGBM+Cat,w={w:.2f})"] = w*pred["LightGBM"] + (1-w)*pred["CatBoost"]

# Ridge stacking
ridge = Ridge(alpha=best_alpha, random_state=SEED)
ridge.fit(meta_X, y_train)
meta_test = np.column_stack([pred[n] for n in meta_names])
pred["StackRidge(CValpha)"] = clip_pred(ridge.predict(meta_test), CAP)

# NNLS stacking
pred["StackNNLS"] = clip_pred(meta_test @ w_nnls, CAP)

rows=[]
for model_name, yhat in pred.items():
    d = metrics(y_test, yhat)
    d["Model"] = model_name
    rows.append(d)

res = pd.DataFrame(rows).sort_values("RMSE").reset_index(drop=True)
display(res.head(15))

best = res.iloc[0].to_dict()
print("\nBEST:", best["Model"], "| RMSE:", best["RMSE"], "| MAE:", best["MAE"], "| R2:", best["R2"])


## 6) Save outputs for reporting

In [ ]:
out_res = f"/content/results_{EXP_NAME}.csv"
res.to_csv(out_res, index=False)
print("Saved:", out_res)

out_cfg = f"/content/config_{EXP_NAME}.json"
cfg = {
    "EXP_NAME": EXP_NAME,
    "TRAIN_PATH": TRAIN_PATH,
    "TEST_PATH": TEST_PATH,
    "SCALER_JSON_PATH": SCALER_JSON_PATH,
    "USE_SCALER_FEATURE_LIST": USE_SCALER_FEATURE_LIST,
    "INCLUDE_EXTRA_COLS": INCLUDE_EXTRA_COLS,
    "CAP": CAP,
    "SEED": SEED,
    "N_SPLITS": N_SPLITS,
    "EARLY_STOP_ROUNDS": EARLY_STOP_ROUNDS,
    "n_features": int(len(features)),
    "features_head": features[:25],
}
with open(out_cfg, "w") as f:
    json.dump(cfg, f, indent=2)
print("Saved:", out_cfg)

# Optional: save best predictions
best_name = res.iloc[0]["Model"]
best_pred = pred[best_name]
pd.DataFrame({"y_true": y_test, "y_pred": best_pred}).to_csv(f"/content/best_pred_{EXP_NAME}.csv", index=False)
print("Saved:", f"/content/best_pred_{EXP_NAME}.csv")
